<a href="https://colab.research.google.com/github/k1-r1ll/pytorch/blob/main/13_pytorch_dataset_dataloader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1.1. Исходный код

In [ ]:
import os
import torchvision
import torchvision.transforms.v2 as tfs

transform = tfs.ToPILImage()

mnist_train = torchvision.datasets.MNIST(r'C:\datasets\mnist', download=True, train=True)
mnist_test = torchvision.datasets.MNIST(r'C:\datasets\mnist', download=True, train=False)

dir_out = 'dataset'
file_format = 'format.json'
train_data = {'dir': 'train', 'data': mnist_train}
test_data = {'dir': 'test', 'data': mnist_test}

if not os.path.exists(dir_out):
    os.mkdir(dir_out)

for info in (train_data, test_data):
    os.mkdir(os.path.join(dir_out, info['dir']))

    for i in range(10):
        os.mkdir(os.path.join(dir_out, info['dir'], f"class_{i}"))

In [ ]:
# 1.2. Апдейт для коллаб

In [ ]:
import os
import torchvision
import torchvision.transforms.v2 as tfs

transform = tfs.ToPILImage()

# ИЗМЕНЕНО: Путь изменен с C:\datasets\mnist на 'mnist' (сохранится внутри Colab)
mnist_train = torchvision.datasets.MNIST('mnist', download=True, train=True)
mnist_test = torchvision.datasets.MNIST('mnist', download=True, train=False)

dir_out = 'dataset'
file_format = 'format.json'
train_data = {'dir': 'train', 'data': mnist_train}
test_data = {'dir': 'test', 'data': mnist_test}

# ИЗМЕНЕНО: Заменено на os.makedirs с exist_ok=True, чтобы код не падал при повторном запуске
if not os.path.exists(dir_out):
    os.makedirs(dir_out, exist_ok=True)

for info in (train_data, test_data):
    # ИЗМЕНЕНО: os.mkdir заменен на os.makedirs
    os.makedirs(os.path.join(dir_out, info['dir']), exist_ok=True)

    for i in range(10):
        # ИЗМЕНЕНО: os.mkdir заменен на os.makedirs
        os.makedirs(os.path.join(dir_out, info['dir'], f"class_{i}"), exist_ok=True)

In [ ]:
# Что изменилось по сравнению со скрином:

# 1. Добавлены import os и import torchvision в самом начале (без них в Colab код выдаст ошибку).
# 2. Строка r'C:\datasets\mnist' заменена на 'mnist'. Теперь файлы скачаются прямо в сессию Colab. После закрытия вкладки они удалятся, не занимая место на вашем компьютере.
# 3. os.mkdir заменен на os.makedirs(..., exist_ok=True). Визуально структура папок останется абсолютно такой же, как на скрине, но теперь вы можете перезапускать ячейку сколько угодно раз — код не будет ругаться, что папка «train» или «class_0» уже существует.
# После запуска этого кода в левой панели Colab (значок папки 📁) появится структура dataset/train/... и dataset/test/....
# Хотите, добавим сюда сохранение самих картинок в эти папки, или в рамках курса вам пока нужно просто подготовить структуру, а сохранять будете позже?


In [ ]:
# 1.3. Все вместе с комментами

In [ ]:
import os  # Библиотека для работы с ОС (создание папок, склейка путей)
import json
from PIL import Image

import torch
import torch.utils.data as data
import torchvision  # Библиотека PyTorch для работы с компьютерным зрением
import torchvision.transforms.v2 as tfs  # Инструмент для трансформации изображений

# Создаем объект преобразования тензоров PyTorch в стандартный формат изображений PIL
transform = tfs.ToPILImage()

# Скачиваем датасет MNIST (рукописные цифры) прямо в виртуальную сессию Colab (в папку 'mnist')
# train=True — скачивает обучающую выборку (60 000 картинок)
mnist_train = torchvision.datasets.MNIST('mnist', download=True, train=True)
# train=False — скачивает тестовую выборку для проверки (10 000 картинок)
mnist_test = torchvision.datasets.MNIST('mnist', download=True, train=False)

dir_out = 'dataset'  # Имя главной (корневой) папки для нашего структурированного датасета
file_format = 'format.json'  # Переменная для формата файла (пока просто объявлена)

# Упаковываем скачанные данные в словари, чтобы потом удобно пройтись по ним в цикле
train_data = {'dir': 'train', 'data': mnist_train}
test_data = {'dir': 'test', 'data': mnist_test}

# Проверяем: если главной папки 'dataset' еще нет, создаем её
# exist_ok=True защищает код от ошибки, если папка уже существует
if not os.path.exists(dir_out):
    os.makedirs(dir_out, exist_ok=True)

# Цикл проходит по очереди по обучающим (train) и тестовым (test) данным
for info in (train_data, test_data):
    # Создаем подпапки: сначала 'dataset/train', затем 'dataset/test'
    os.makedirs(os.path.join(dir_out, info['dir']), exist_ok=True)

    # Внутренний цикл: генерирует числа от 0 до 9 для создания папок под каждый класс цифр
    for i in range(10):
        # Создаем папки для классов (например: 'dataset/train/class_0', 'dataset/train/class_1' и т.д.)
        os.makedirs(os.path.join(dir_out, info['dir'], f"class_{i}"), exist_ok=True)


In [ ]:
class DigitDataset(data.Dataset):
  def __init__(self, path, train=True, transform=None):
    self.path = os.path.join(path, "train" if train else "test")
    self.transform = transform

    with open(os.path.join(path, "format.json"), "r") as fp:
      self.format = json.load(fp)

    self.length = 0 # размер обучающей выборки
    self.files = [] # в итоге это будет список кортежей, один кортеж -- это путь к файлу и его метка
    self.targets = torch.eye(10) # целевые векторы (на выходе), в виде единичной матрицы

    # Перебираем пары: имя подпапки (_dir) и его целевое назначение или формат (_target) в сохраненном джейсоне
    # Программа смотрит в настройки (в джейсоне, кажется) и видит: "Ага, у нас есть папка cats (её цель — определять кошек) и папка dogs (цель — собаки)".
    for _dir, _target in self.format.items():

        # Формируем полный путь к текущей папке, объединяя базовый путь и имя подкаталога
        # Строит точный путь на диске: C:/my_project/dataset/cats
        path = os.path.join(self.path, _dir)
        # Получаем список имён всех файлов и папок, находящихся в этом каталоге
        # Сканирует эту папку и получает список имен файлов в подпапке: ['cat1.jpg', 'cat2.jpg', 'cat3.jpg'...]
        list_files = os.listdir(path)
        # Прибавляем количество найденных файлов к общему счётчику self.length
        self.length += len(list_files)
        # Для каждого файла создаем кортеж (полный_путь_к_файлу, целевой_формат)
        # и добавляем все эти кортежи в общий список self.files
        # Она берет имя каждого файла, склеивает его с путем к папке и его меткой, а затем записывает в один длинный общий список self.files.
        # В итоге после работы этого цикла в памяти компьютера появляется готовый список-каталог:
        # self.files = [
                  #     ("C:/my_project/dataset/cats/cat1.jpg", "cats"),
                  #     ("C:/my_project/dataset/cats/cat2.jpg", "cats"),
                  #     ...
                  #     ("C:/my_project/dataset/dogs/dog1.jpg", "dogs"),
                  #     ("C:/my_project/dataset/dogs/dog2.jpg", "dogs"),
                  # ]
        self.files.extend(map(lambda _x: (os.path.join(path, _x), _target), list_files))

# Магический метод для получения элемента по индексу (например, dataset[i])
def __getitem__(self, item):

    # Извлекаем путь к файлу и его текстовую метку/класс из списка по индексу item
    path_file, target = self.files[item]
    # Получаем закодированную метку (например, числовой ID класса) из словаря targets
    t = self.targets[target]
    # Открываем изображение по его полному пути с помощью библиотеки PIL (Pillow)
    img = Image.open(path_file)
    # Возвращаем кортеж из открытого изображения и его метки класса
    if self.transform:
      img = self.transform(img).ravel().float() / 255.0 # 255.0 -- это нормирование, максимальное значение пикселя

    return img, t

def __len__(self):
  return self.length

# Давайте представим, что вы вызвали датасет[0] (хотите получить самую первую картинку):
# 1. def __getitem__(self, item):
# 	• В переменную item прилетает число 0 (наш индекс).
# 2. path_file, target = self.files[item]
# 	• Мы лезем в наш внутренний блокнот self.files под номером 0.
# 	• Там написано: ("папка/кошка.jpg", "кошка").
# 	• Мы раскладываем это по полочкам: в path_file кладём адрес картинки, а в target — слово "кошка".
# 3. t = self.targets[target]
# 	• Компьютер не понимает слово "кошка", ему нужны цифры (0, 1, 2...).
# 	• Мы смотрим в другой словарик: self.targets = {"кошка": 0, "собака": 1}.
# 	• Ищем там слово "кошка" и получаем цифру 0. Записываем её в t.
# 4. img = Image.open(path_file)
# 	• Мы берём адрес "папка/кошка.jpg", компьютер идёт по этому адресу на жесткий диск, открывает файл и загружает саму картинку в оперативную память под именем img.
# 5. return img, t
# 	• Функция отдаёт вам готовый результат: саму открытую картинку и её цифровой код 0.

In [ ]:
# 2.1. Рабочий код для коллаба, с дополнением в виде непосредственно сохранения картинок по директориям

In [ ]:
import os
import json
from PIL import Image

import torch
import torch.utils.data as data
import torchvision
import torchvision.transforms.v2 as tfs

transform = tfs.ToPILImage()

mnist_train = torchvision.datasets.MNIST('mnist', download=True, train=True)
mnist_test = torchvision.datasets.MNIST('mnist', download=True, train=False)

dir_out = 'dataset'
file_format = 'format.json'
train_data = {'dir': 'train', 'data': mnist_train}
test_data = {'dir': 'test', 'data': mnist_test}

if not os.path.exists(dir_out):
    os.makedirs(dir_out, exist_ok=True)

for info in (train_data, test_data):
    os.makedirs(os.path.join(dir_out, info['dir']), exist_ok=True)

    print(f"Начинаю сохранять картинки для {info['dir']}...")

    # А вот этот цикл у вас потерялся! Пробегаемся по всем картинкам датасета
    for index, (img, label) in enumerate(info['data']):
        # Формируем путь для сохранения, например: dataset/train/class_5/img_0.png
        filename = f"img_{index}.png"
        path_to_save = os.path.join(dir_out, info['dir'], f"class_{label}", filename)
        # Переводим тензор/данные в картинку PIL (если нужно) и сохраняем
        if not isinstance(img, Image.Image):
            img = transform(img)

        img.save(path_to_save)

    print("🎉 Все картинки успешно сохранены по папкам!")

    for i in range(10):
        os.makedirs(os.path.join(dir_out, info['dir'], f"class_{i}"), exist_ok=True)

In [1]:
# 2.2. Рабочий код для коллаба, с дополнением в виде непосредственно сохранения картинок: что и выше, но с ограничением

In [1]:
import os
import json
from PIL import Image

import torch
import torch.utils.data as data
import torchvision
import torchvision.transforms.v2 as tfs

transform = tfs.ToPILImage()

mnist_train = torchvision.datasets.MNIST('mnist', download=True, train=True)
mnist_test = torchvision.datasets.MNIST('mnist', download=True, train=False)

dir_out = 'dataset'
file_format = 'format.json'

# Добавили лимиты прямо в настройки данных
train_data = {'dir': 'train', 'data': mnist_train, 'limit': 200}
test_data = {'dir': 'test', 'data': mnist_test, 'limit': 50}

if not os.path.exists(dir_out):
    os.makedirs(dir_out, exist_ok=True)

for info in (train_data, test_data):
    # Создаем базовую папку (train или test)
    os.makedirs(os.path.join(dir_out, info['dir']), exist_ok=True)

    # Перенесли создание папок class_0...class_9 наверх,
    # чтобы они уже были готовы перед сохранением картинок
    for i in range(10):
        os.makedirs(os.path.join(dir_out, info['dir'], f"class_{i}"), exist_ok=True)

    print(f"Начинаю сохранять картинки для {info['dir']} (лимит: {info['limit']})...")

    for index, (img, label) in enumerate(info['data']):
        # Проверяем лимит: если сохранили нужное количество, останавливаем цикл
        if index >= info['limit']:
            break

        filename = f"img_{index}.png"
        path_to_save = os.path.join(dir_out, info['dir'], f"class_{label}", filename)
        if not isinstance(img, Image.Image):
            img = transform(img)

        img.save(path_to_save)

    print(f"🎉 Картинки для {info['dir']} успешно сохранены в количестве {info['limit']} шт.!")


100%|██████████| 9.91M/9.91M [00:00<00:00, 119MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 9.21MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 27.2MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 7.42MB/s]


Начинаю сохранять картинки для train (лимит: 200)...
🎉 Картинки для train успешно сохранены в количестве 200 шт.!
Начинаю сохранять картинки для test (лимит: 50)...
🎉 Картинки для test успешно сохранены в количестве 50 шт.!


In [2]:
# 2.3. Немного доделанный __getitem__

In [32]:
class DigitDataset(data.Dataset):
    def __init__(self, path, train=True, transform=None):
        self.path = os.path.join(path, "train" if train else "test")
        self.transform = transform

        with open(os.path.join(path, "format.json"), "r") as fp:
            self.format = json.load(fp)

        self.length = 0
        self.files = []
        self.targets = torch.eye(10)

        for _dir, _target in self.format.items():
            path_dir = os.path.join(self.path, _dir)
            if not os.path.exists(path_dir):
                continue
            list_files = os.listdir(path_dir)
            self.length += len(list_files)
            self.files.extend(map(lambda _x: (os.path.join(path_dir, _x), _target), list_files))

    def __getitem__(self, item):
        path_file, target = self.files[item]

        # Защита: приводим имя класса к виду "class_4"
        target_str = f"class_{target}" if not str(target).startswith("class_") else str(target)

        # Берем ID из формата и достаем One-Hot вектор (сначала метку из джейсона, а потом соответствующий вектор с единицей среди нулей)
        target_id = self.format[target_str]
        t = self.targets[target_id]

        img = Image.open(path_file)
        if self.transform:
            # Превращаем картинку в тензор с помощью tfs
            img_tensor = self.transform(img)

            # Вытягиваем в одномерный вектор (ravel) и переводим в float
            # Деление на 255 убираем, так как tfs.ToImage() / ToDtype сами приводят к диапазону [0, 1]
            img = img_tensor.ravel().float() / 255

        return img, t

    def __len__(self):
        return self.length

In [22]:
to_tensor = tfs.ToImage()
d_train = DigitDataset("dataset", transform=to_tensor)
train_data = data.DataLoader(d_train, batch_size=32, shuffle=True)

In [23]:
print(len(d_train))

200


In [ ]:
#it = iter(train_data)
#x, y = next(it)
#x.shape
#y.shape

# Если разложить по полочкам, то у каждого из этих двух инструментов своя четкая работа:
# • DigitDataset — это «каталог и доставщик». Сначала он один раз переписывает все адреса картинок и их правильные ответы, чтобы не держать тяжелые изображения в памяти. А потом по запросу находит нужный файл, открывает его, превращает в плоский числовой вектор и выдает вместе с ответом.
# • DataLoader — это «упаковщик и конвейер». Он берет доставщика за руку, перемешивает картинки в случайном порядке, собирает их в аккуратные пачки (батчи по 32 штуки) и ровным потоком подает на вход нейросети для обучения.
# Ты разобрался в одной из самых фундаментальных концепций PyTorch!
# Какую строчку или деталь из этого механизма разберем следующей?

# {
#   "class_0": 0,
#   "class_1": 1,
#   "class_2": 2,
#   "class_3": 3,
#   "class_4": 4,
#   "class_5": 5,
#   "class_6": 6,
#   "class_7": 7,
#   "class_8": 8,
#   "class_9": 9
# }


In [24]:
import torch.nn as nn
import torch.optim as optim

In [25]:
class DigitNN(nn.Module):
  def __init__(self, imp_dim, hid_dim, out_dim):
      super().__init__()
      self.l1 = nn.Linear(imp_dim, hid_dim)
      self.l2 = nn.Linear(hid_dim, out_dim)

  def forward(self, x):
    x = self.l1(x)
    x = nn.functional.relu(x)
    x = self.l2(x)
    return x


In [26]:
model = DigitNN(28 * 28, 32, 10)
optimizer = optim.Adam(params=model.parameters(), lr=0.01)
loss_func = nn.CrossEntropyLoss()
epochs = 2
model.train()

DigitNN(
  (l1): Linear(in_features=784, out_features=32, bias=True)
  (l2): Linear(in_features=32, out_features=10, bias=True)
)

In [27]:
for _e in range(epochs):
  for x_train, y_train in train_data:
    predict = model(x_train)
    loss = loss_func(predict, y_train)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

In [28]:
d_test = DigitDataset("dataset", train=False, transform=to_tensor)
test_data = data.DataLoader(d_test, batch_size=500, shuffle=False)
Q = 0

In [29]:
model.eval()

DigitNN(
  (l1): Linear(in_features=784, out_features=32, bias=True)
  (l2): Linear(in_features=32, out_features=10, bias=True)
)

In [30]:
for x_test, y_test in test_data:
  with torch.no_grad():
    p = model(x_test)
    p = torch.argmax(p, dim=1)
    y = torch.argmax(y_test, dim=1)
    Q += torch.sum(p == y).item()

In [31]:
Q /= len(d_test)
print(Q)

0.46


In [1]:
import os
import json
from PIL import Image

import torch
import torch.utils.data as data
import torchvision
import torchvision.transforms.v2 as tfs

import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm

transform = tfs.ToPILImage()

mnist_train = torchvision.datasets.MNIST('mnist', download=True, train=True)
mnist_test = torchvision.datasets.MNIST('mnist', download=True, train=False)

dir_out = 'dataset'
file_format = 'format.json'

# Добавили лимиты прямо в настройки данных
train_data = {'dir': 'train', 'data': mnist_train}
test_data = {'dir': 'test', 'data': mnist_test}

if not os.path.exists(dir_out):
    os.makedirs(dir_out, exist_ok=True)

for info in (train_data, test_data):
    os.makedirs(os.path.join(dir_out, info['dir']), exist_ok=True)

    for i in range(10):
        os.makedirs(os.path.join(dir_out, info['dir'], f"class_{i}"), exist_ok=True)

    print(f"Начинаю сохранять картинки для {info['dir']}")

    for index, (img, label) in enumerate(info['data']):

        filename = f"img_{index}.png"
        path_to_save = os.path.join(dir_out, info['dir'], f"class_{label}", filename)
        if not isinstance(img, Image.Image):
            img = transform(img)

        img.save(path_to_save)

    print(f"Картинки для {info['dir']} успешно сохранены")


100%|██████████| 9.91M/9.91M [00:00<00:00, 20.2MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 501kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.70MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 8.23MB/s]


Начинаю сохранять картинки для train
Картинки для train успешно сохранены
Начинаю сохранять картинки для test
Картинки для test успешно сохранены


In [23]:
class DigitDataset(data.Dataset):
    def __init__(self, path, train=True, transform=None):
        self.path = os.path.join(path, "train" if train else "test")
        self.transform = transform

        with open(os.path.join(path, "format.json"), "r") as fp:
            self.format = json.load(fp)

        self.length = 0
        self.files = []
        self.targets = torch.eye(10)

        for _dir, _target in self.format.items():
            path_dir = os.path.join(self.path, _dir)
            if not os.path.exists(path_dir):
                continue
            list_files = os.listdir(path_dir)
            self.length += len(list_files)
            self.files.extend(map(lambda _x: (os.path.join(path_dir, _x), _target), list_files))

    def __getitem__(self, item):
        path_file, target = self.files[item]

        # Защита: приводим имя класса к виду "class_4"
        target_str = f"class_{target}" if not str(target).startswith("class_") else str(target)

        # Берем ID из формата и достаем One-Hot вектор (сначала метку из джейсона, а потом соответствующий вектор с единицей среди нулей)
        target_id = self.format[target_str]
        t = self.targets[target_id]

        img = Image.open(path_file)
        if self.transform:
            img_tensor = self.transform(img)
            img = img_tensor.ravel().float() / 255

        return img, t

    def __len__(self):
        return self.length

In [24]:
to_tensor = tfs.ToImage()
d_train = DigitDataset("dataset", transform=to_tensor)
train_data = data.DataLoader(d_train, batch_size=32, shuffle=True)

In [25]:
class DigitNN(nn.Module):
  def __init__(self, imp_dim, hid_dim, out_dim):
      super().__init__()
      self.l1 = nn.Linear(imp_dim, hid_dim)
      self.l2 = nn.Linear(hid_dim, out_dim)

  def forward(self, x):
    x = self.l1(x)
    x = nn.functional.relu(x)
    x = self.l2(x)
    return x

In [26]:
model = DigitNN(28 * 28, 32, 10)
optimizer = optim.Adam(params=model.parameters(), lr=0.01)
loss_func = nn.CrossEntropyLoss()
epochs = 2
model.train()

DigitNN(
  (l1): Linear(in_features=784, out_features=32, bias=True)
  (l2): Linear(in_features=32, out_features=10, bias=True)
)

In [36]:
for _e in range(epochs):
  loss_mean = 0
  lm_count = 0


  train_tqdm = tqdm(train_data, leave=True)

  for x_train, y_train in train_tqdm:
    predict = model(x_train)
    loss = loss_func(predict, y_train)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    lm_count += 1

    # • Зачем это нужно: Во время обучения нейросети ошибка постоянно скачет от батча к батчу. Чтобы не копить в памяти список из тысяч чисел и не делить их сумму в конце, среднее обновляют плавно на каждом новом шаге.
    # • lm_count: Это обычный счетчик шагов (сколько батчей уже прошло с начала эпохи). Чуть выше на картинке видно, что он каждый раз увеличивается на единицу.
    # • loss.item(): Это чистое число (значение ошибки) на текущем шаге, которое мы достаем из тензора PyTorch.
    # • Логика весов: Формула берет старое среднее значение (loss_mean) с большим весом и добавляет к нему новую ошибку (loss.item()) с маленьким весом. Чем больше шагов прошло, тем меньше каждый отдельный новый шаг влияет на общее среднее.
    # Эта строчка вообще не участвует в обучении нейросети и не меняет её веса.
    # • Её единственная задача: Посчитать обычное среднее арифметическое всех ошибок (loss) за эпоху. Она просто берет старое среднее, аккуратно подмешивает в него новую ошибку и сохраняет результат для статистики, чтобы в конце эпохи вывести красивую среднюю цифру в консоль или на график.
    loss_mean = 1 / lm_count * loss.item() + (1 - 1 / lm_count) * loss_mean
    train_tqdm.set_description(f"Epoch {_e + 1}/{epochs}, loss_mean: {loss_mean}")


Epoch 2/2, loss_mean: 0.1288372465499984: 100%|██████████| 1875/1875 [00:26<00:00, 69.59it/s]


In [29]:
d_test = DigitDataset("dataset", train=False, transform=to_tensor)
test_data = data.DataLoader(d_test, batch_size=500, shuffle=False)
Q = 0

In [30]:
model.eval()

DigitNN(
  (l1): Linear(in_features=784, out_features=32, bias=True)
  (l2): Linear(in_features=32, out_features=10, bias=True)
)

In [31]:
for x_test, y_test in test_data:
  with torch.no_grad():
    p = model(x_test)
    p = torch.argmax(p, dim=1)
    y = torch.argmax(y_test, dim=1)
    Q += torch.sum(p == y).item()

In [32]:
Q /= len(d_test)
print(Q)

0.9485


In [33]:
from tqdm import tqdm